## Data Visualization

### Loading Libraries

In [1]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [2]:
OUTPUT_DIR = "../data/citibike/"

In [4]:
citibike_df = pd.read_csv("../data/citibike/JC/JC2025_Enriched.csv")

In [ ]:
weather_daily = pd.read_csv(f'{output_dir}/JC/jersey_weather_2025.csv', parse_dates=['date'])
weather_daily.info()

### Visualizations based on Monthly Data

In [6]:
monthly_rides = (
    citibike_df
    .groupby("month", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

monthly_rides

,month,number_of_rides
0,2025-01,6
1,2025-02,270786
2,2025-03,438744
3,2025-04,487770
4,2025-05,557280
5,2025-06,580416
6,2025-07,644244
7,2025-08,648006
8,2025-09,693473
9,2025-10,540876


In [7]:
fig = px.bar(
    monthly_rides,
    x="month",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Month"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Rides",
)

fig.show()

In [8]:
# Number of Rides per Season
season_rides = (
    citibike_df
    .groupby("season", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

season_order = ["Winter", "Spring", "Summer", "Autumn"]

season_rides["season"] = pd.Categorical(
    season_rides["season"],
    categories=season_order,
    ordered=True
)

season_rides = season_rides.sort_values("season")

season_rides

,season,number_of_rides
3,Winter,510102
1,Spring,1483794
2,Summer,1872666
0,Autumn,1615514


In [9]:
fig = px.bar(
    season_rides,
    x="season",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Season",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Season",
    yaxis_title="Number of Rides"
)

fig.show()

In [ ]:
# Usually, warmer seasons are expected to have more rides, while winter may have fewer rides due to colder weather, snow, wind, and shorter daylight hours.

In [10]:
# Top 10 Start Stations
top_start_stations = (
    citibike_df
    .dropna(subset=["start_station_name"])
    .groupby("start_station_name", as_index=False)
    .agg(
        number_of_departures=("ride_id", "count")
    )
    .sort_values("number_of_departures", ascending=False)
    .head(10)
)

top_start_stations

,start_station_name,number_of_departures
52,Grove St PATH,244456
58,Hoboken Terminal - Hudson St & Hudson Pl,142558
53,Hamilton Park,122260
93,River St & Newark St,121027
84,Newport PATH,113275
44,Exchange Pl,110247
18,Bergen Ave & Sip Ave,110222
0,11 St & Washington St,107318
85,Newport Pkwy,103976
92,River St & 1 St,103948


In [11]:
fig = px.bar(
    top_start_stations.sort_values("number_of_departures"),
    x="number_of_departures",
    y="start_station_name",
    orientation="h",
    title="Top 10 Start Stations by Number of Departures",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Departures",
    yaxis_title="Start Station"
)

fig.show()

In [ ]:
# High departure volume may indicate stations near residential areas, transit hubs, offices, universities, or busy public locations.

In [12]:
# Top 10 End Stations
top_end_stations = (
    citibike_df
    .dropna(subset=["end_station_name"])
    .groupby("end_station_name", as_index=False)
    .agg(
        number_of_arrivals=("ride_id", "count")
    )
    .sort_values("number_of_arrivals", ascending=False)
    .head(10)
)

top_end_stations

,end_station_name,number_of_arrivals
229,Grove St PATH,259946
238,Hoboken Terminal - Hudson St & Hudson Pl,146358
340,River St & Newark St,124079
230,Hamilton Park,122935
311,Newport PATH,114186
71,Bergen Ave & Sip Ave,110771
204,Exchange Pl,110292
7,11 St & Washington St,107247
312,Newport Pkwy,103549
10,14 St Ferry - 14 St & Shipyard Ln,102192


In [13]:
fig = px.bar(
    top_end_stations.sort_values("number_of_arrivals"),
    x="number_of_arrivals",
    y="end_station_name",
    orientation="h",
    title="Top 10 End Stations by Number of Arrivals",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Arrivals",
    yaxis_title="End Station"
)

fig.show()

In [ ]:
# These stations are the most common destinations of Citi Bike rides.

# High arrival volume may indicate stations near workplaces, shopping areas, public transportation, parks, or popular city destinations.

In [14]:
# Merge with Weather Data to See Patterns
daily_rides = (
    citibike_df
    .groupby("date", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)
daily_rides["date"] = pd.to_datetime(daily_rides["date"])
daily_rides.head()

,date,number_of_rides
0,2025-01-31,6
1,2025-02-01,8610
2,2025-02-02,6546
3,2025-02-03,10746
4,2025-02-04,12342


In [19]:
weather_daily.head()


,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max,date
0,10.9,3.9,7.4,4.5,4.5,0.0,23.2,2025-01-01
1,5.4,0.3,2.6,0.0,0.0,0.0,25.1,2025-01-02
2,3.2,-1.9,0.4,0.0,0.0,0.0,17.1,2025-01-03
3,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1,2025-01-04
4,0.3,-3.6,-2.2,0.0,0.0,0.0,19.9,2025-01-05


In [20]:
# merge the two datasets
bike_weather_daily = daily_rides.merge(
    weather_daily,
    on="date",
    how="left"
)

bike_weather_daily.head()

,date,number_of_rides,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2025-01-31,6,6.2,0.7,3.6,10.3,10.3,0.00,13.5
1,2025-02-01,8610,6.2,-5.8,0.7,2.4,2.4,0.00,19.4
2,2025-02-02,6546,1.4,-9.6,-4.5,1.2,0.0,0.84,11.7
3,2025-02-03,10746,6.1,-3.0,1.9,0.3,0.0,0.21,12.8
4,2025-02-04,12342,6.2,-0.5,3.6,0.3,0.3,0.00,23.0


In [ ]:
# Check missing values after the merge
bike_weather_daily.isna().sum()

date                   0
number_of_rides        0
temperature_2m_max     0
temperature_2m_min     0
temperature_2m_mean    0
precipitation_sum      0
rain_sum               0
snowfall_sum           0
wind_speed_10m_max     0
dtype: int64

In [29]:
# Rides and Average Temperature
fig = px.scatter(
    bike_weather_daily,
    x="temperature_2m_mean",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Average Temperature"
)

fig.update_layout(
    xaxis_title="Average Daily Temperature",
    yaxis_title="Number of Rides"
)

fig.show()

In [30]:
# Rides and Wind Speed
fig = px.scatter(
    bike_weather_daily,
    x="wind_speed_10m_max",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Maximum Wind Speed"
)

fig.update_layout(
    xaxis_title="Maximum Wind Speed",
    yaxis_title="Number of Rides"
)

fig.show()

In [31]:
# Rides and Precipitation
fig = px.scatter(
    bike_weather_daily,
    x="precipitation_sum",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Precipitation"
)

fig.update_layout(
    xaxis_title="Daily Precipitation",
    yaxis_title="Number of Rides"
)

fig.show()

### Rides vs temperature

In [32]:
import plotly.graph_objects as go


fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["number_of_rides"],
        mode="lines",
        name="Daily Rides",
        yaxis="y1"
    )
)

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["temperature_2m_mean"],
        mode="lines",
        name="Daily Average Temperature",
        yaxis="y2"
    )
)

fig.update_layout(
    title="Daily Rides and Daily Average Temperature",
    xaxis=dict(
        title="Day"
    ),
    yaxis=dict(
        title="Daily Rides",
        side="left"
    ),
    yaxis2=dict(
        title="Daily Average Temperature",
        overlaying="y",
        side="right"
    ),
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()